# Aggregating the w2 prediction results
In this notebook, we will aggregate the wasserstein distance results across all the models.

Unlike fate accuracy, the starting cells are all staying in HSC states at the first time point (day2). In this task, the starting cells can be either HSPC or commited progenitors at day4. The starting cells are all held out during training. We then measure their distance to their descendent both in population wise (Well=2) and clone wise (92 clones). 

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import json

In [103]:
os.chdir("/rds/user/wz369/hpc-work/pseudodynamics_plus")
higher_dir = '/rds/user/wz369/hpc-work'

# pseudodynamics+ (pdp+)

## PC 30

In [3]:
import re
def w2_file_regex(filename):
    patterns = re.match(r"t(\S{,3})_n(\S{,3})_(\S{0,3})_w2_eval.csv", filename).groups()
    return patterns

In [89]:
eval_dir = f"{higher_dir}/pseudodynamics_plus/results/pseudodynamics+/" 

PCconfig_repeats = [
    "klein_PC_30_lD1_lv1_lgNone",
    "klein_PC30_lD1_cfm1_lgNone",
    "klein_PC30_lD1_cfm1_lv1_lgNone",
    "klein_PC30_lD1_cfm2_lgNone",
    "klein_PC30_lD1_cfm10_lgNone",
    "klein_PC30_lD1_cfm10_lgNone_b1024"
    ]

# loops through all the eval files
results = []
for repeat in PCconfig_repeats:
    eval_subfolder = os.path.join(eval_dir, repeat)
    for file in os.listdir(eval_subfolder):
        if file.endswith("_w2_eval.csv"):
            int_time, noise, sim_fn = w2_file_regex(file)
            # print(int_time, noise, sim_fn)
            df = pd.read_csv(os.path.join(eval_subfolder, file))
            # df['sim_fn'] = sim_fn
            if "w2_scaled" not in df.columns:
                continue
            else:
                results.append(df)
            
# drop the intemediate simulation results
pdp_eval_results = pd.concat(results)

In [90]:
pdp_PC = pdp_eval_results.sort_values(
    by=['w2_raw'], ascending=True
        ).drop_duplicates(subset=['model_dir'], keep='first')
pdp_PC['space'] = 'PC30'
pdp_PC['method'] = 'pseudodynamics+'

In [91]:
pdp_PC

,w2_scaled,w2_raw,n_sims,t_end_norm,noise_scale,sim_fn,n_steps,model_dir,unstandardized,space,method
0,3.944441,11.125337,10,2.5,1.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,pseudodynamics+
0,3.963885,11.219783,10,2.5,1.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,pseudodynamics+
0,4.049274,11.323343,10,3.0,1.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,pseudodynamics+
0,3.923314,11.345748,10,4.0,1.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,pseudodynamics+
0,4.037858,11.645900,10,3.0,1.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,pseudodynamics+
0,4.032357,11.723080,10,3.5,2.0,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,PC30,pseudodynamics+


## Diffusion map (10 dimension)

In [85]:
eval_dir = f"{higher_dir}/pseudodynamics_plus/results/pseudodynamics+/" 

PCconfig_repeats = [
    "klein_DMscaled_10_cfm5_b512",
    "klein_DMscaled_10_cfm5_b1024",
    "klein_DMscaled_10_cfm10_b512",
    "klein_DMscaled_10_cfm10_b1024",
    ]

# loops through all the eval files
results = []
for repeat in PCconfig_repeats:
    eval_subfolder = os.path.join(eval_dir, repeat)
    for file in os.listdir(eval_subfolder):
        if file.endswith("_w2_eval.csv"):
            int_time, noise, sim_fn = w2_file_regex(file)
            # print(int_time, noise, sim_fn)
            df = pd.read_csv(os.path.join(eval_subfolder, file))
            df['sim_fn'] = sim_fn
            if "w2_scaled" not in df.columns:
                continue
            else:
                results.append(df)
            
# drop the intemediate simulation results
pdp_eval_results = pd.concat(results)
pdp_DM = pdp_eval_results.sort_values(
    by=['w2_scaled'], ascending=True
        ).drop_duplicates(subset=['model_dir'], keep='first')
pdp_DM['space'] = 'DM10'
pdp_DM['method'] = 'pseudodynamics+'
pdp_DM

,w2_scaled,w2_raw,n_sims,t_end_norm,noise_scale,sim_fn,n_steps,model_dir,unstandardized,space,method
0,2.624942,0.007227,10,2.5,1.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,DM10,pseudodynamics+
0,2.634721,0.007252,10,2.5,1.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,DM10,pseudodynamics+
0,2.655129,0.007323,10,2.5,1.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,DM10,pseudodynamics+
0,2.666888,0.007353,10,2.5,1.5,sde,200,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,True,DM10,pseudodynamics+


# DeepRUOT

In [35]:
eval_dir = f"{higher_dir}/DeepRUOTv2/results/"
print(eval_dir)

/rds/user/wz369/hpc-work/DeepRUOTv2/results/


In [10]:
PC_repeats = [
    'klein_pca30', 'klein_pca30_s0', 'klein_pca30_s10', 'klein_pca30_s42'
]

DeepRUOTresults = []
for repeat in PC_repeats:
    print(repeat)
    pf = pd.read_csv(os.path.join(eval_dir, repeat, "eval_combined.csv"))
    
    DeepRUOTresults.append(pf)

DeepRUOT_pc = pd.concat(DeepRUOTresults) 
DeepRUOT_pc['space'] = 'PC30'
DeepRUOT_pc['method'] = 'DeepRUOTv2'
DeepRUOT_pc

klein_pca30
klein_pca30_s0
klein_pca30_s10
klein_pca30_s42


,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.360906,0.912658,11.860204,11.860204,2031,1,15,PC30,DeepRUOTv2
0,ode,0.334810,0.975341,11.517741,11.517741,2031,1,15,PC30,DeepRUOTv2
0,ode,0.341704,0.822840,11.682735,11.682735,2031,1,15,PC30,DeepRUOTv2
0,ode,0.324471,0.646240,11.315379,11.315379,2031,1,15,PC30,DeepRUOTv2


In [11]:
DMrepeats = [
    'klein_dm10', 'klein_dm10_s0', 'klein_dm10_s10', 'klein_dm10_s42'
]

DeepRUOTresults = []
for repeat in DMrepeats:
    print(repeat)
    pf = pd.read_csv(os.path.join(eval_dir, repeat, "eval_combined.csv"))
    
    DeepRUOTresults.append(pf)

DeepRUOT_dm = pd.concat(DeepRUOTresults) 
DeepRUOT_dm['space'] = 'DM10'
DeepRUOT_dm['method'] = 'DeepRUOTv2'
DeepRUOT_dm

klein_dm10
klein_dm10_s0
klein_dm10_s10
klein_dm10_s42


,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.076809,0.809221,0.006889,0.006889,2031,1,15,DM10,DeepRUOTv2
0,ode,0.164451,0.731430,0.007299,0.007299,2031,1,15,DM10,DeepRUOTv2
0,ode,0.084687,0.815819,0.007517,0.007517,2031,1,15,DM10,DeepRUOTv2
0,ode,0.147710,0.739281,0.006890,0.006890,2031,1,15,DM10,DeepRUOTv2


# scDiffeq

In [12]:
results = []
for seed in [0,1,2]:
    eval_dir = f"{higher_dir}/scDiffEq/results/seeds/seed_{seed}/pca30/plain_sde/fate_prediction_metrics/last"
    df = pd.read_csv(os.path.join(eval_dir, "eval_combined.csv"))
    results.append(df)

scDiffeq_pc = pd.concat(results)
scDiffeq_pc['space'] = 'PC30' 
scDiffeq_pc['method'] = 'scDiffEq'
scDiffeq_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,sde,0.385032,0.846869,10.490688,10.490688,2031,200,15,PC30,scDiffEq
0,sde,0.377646,0.846252,10.662208,10.662208,2031,200,15,PC30,scDiffEq
0,sde,0.373215,0.840821,10.662832,10.662832,2031,200,15,PC30,scDiffEq


In [13]:
results = []
for seed in [0,1,2]:
    eval_dir = f"{higher_dir}/scDiffEq/results/seeds/seed_{seed}/dm10/plain_sde/fate_prediction_metrics/last"
    df = pd.read_csv(os.path.join(eval_dir, "eval_combined.csv"))
    results.append(df)

scDiffeq_dm = pd.concat(results)
scDiffeq_dm['space'] = 'PC30' 
scDiffeq_dm['method'] = 'scDiffEq'
scDiffeq_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,sde,0.434761,0.734353,0.007385,0.007385,2031,200,15,PC30,scDiffEq
0,sde,0.274742,0.908217,0.007252,0.007252,2031,200,15,PC30,scDiffEq
0,sde,0.383555,0.777750,0.007415,0.007415,2031,200,15,PC30,scDiffEq


# TIGON
To run evaluation :

```shell
bash /rds/user/wz369/hpc-work/PINN_dynamics/scripts/TIGON/03_evaluate.sh
```

In [14]:
results = []
for repeat in ["model", "model_seed2", "model_seed3"]:
    df = pd.read_csv(f"{higher_dir}/PINN_dynamics/logs/TIGON/pca/{repeat}/eval_combined.csv")
    results.append(df)
TIGON_pc = pd.concat(results)
TIGON_pc['space'] = 'PC30'
TIGON_pc['method'] = 'TIGON'
TIGON_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.038897,0.980651,4.251537,12.15261,2031,1,15,PC30,TIGON
0,ode,0.038897,0.982976,4.250278,12.14623,2031,1,15,PC30,TIGON
0,ode,0.038897,0.982976,4.250278,12.14623,2031,1,15,PC30,TIGON


In [15]:
results = []
for repeat in ["model", "model_seed2", "model_seed3"]:
    df = pd.read_csv(f"{higher_dir}/PINN_dynamics/logs/TIGON/dm/{repeat}/eval_combined.csv")
    results.append(df)
TIGON_dm = pd.concat(results)
TIGON_dm['space'] = 'DM10'
TIGON_dm['method'] = 'TIGON'
TIGON_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.078779,0.873658,2.837012,0.007061,2031,1,15,DM10,TIGON
0,ode,0.036435,0.963030,2.803302,0.006978,2031,1,15,DM10,TIGON
0,ode,0.036435,0.963030,2.803302,0.006978,2031,1,15,DM10,TIGON


# MIOFlow

In [16]:
results = []
for repeat in [1,2,3]:
    df = pd.read_csv(f"logs/MIOFlow/klein_pca_gaga_run{repeat}/eval_combined.csv")
    # df = pd.read_csv(f"logs/MIOFlow/klein_pca_gaga_latent15_run{repeat}/eval_combined.csv")
    results.append(df)
mioflow_pc = pd.concat(results)
mioflow_pc['space'] = 'PC30'
mioflow_pc['method'] = 'MIOFLow'
mioflow_pc

,sim_mode,accuracy,pearson_r,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.179714,0.954438,13.410164,2031,1,15,PC30,MIOFLow
0,ode,0.157065,0.218102,12.622305,2031,1,15,PC30,MIOFLow
0,ode,0.284097,0.826622,13.920834,2031,1,15,PC30,MIOFLow


In [17]:
results = []
for repeat in [1,2,3]:
    df = pd.read_csv(f"logs/MIOFlow/klein_dm_gaga_run{repeat}/eval_combined.csv")
    # df = pd.read_csv(f"logs/MIOFlow/klein_pca_gaga_latent15_run{repeat}/eval_combined.csv")
    results.append(df)
mioflow_dm = pd.concat(results)
mioflow_dm['space'] = 'DM10'
mioflow_dm['method'] = 'MIOFlow'
mioflow_dm

,sim_mode,accuracy,pearson_r,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.123584,0.057220,0.024957,2031,1,15,DM10,MIOFlow
0,ode,0.103890,0.065061,0.018648,2031,1,15,DM10,MIOFlow
0,ode,0.290005,0.497005,0.014257,2031,1,15,DM10,MIOFlow


# TrajectoryNet

In [18]:
results = []
for repeat in ['model','model_run2','model_run3','model_run4']:
    df = pd.read_csv(f"logs/TrajectoryNet/pca30/{repeat}/eval_combined.csv")
    results.append(df)

TJN_pc = pd.concat(results)
TJN_pc['space'] = 'PC30'
TJN_pc['method'] = 'TrajectoryNet'
TJN_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.092565,0.780078,4.696540,12.383142,2031,1,15,PC30,TrajectoryNet
0,ode,0.084195,0.823549,4.762419,12.415095,2031,1,15,PC30,TrajectoryNet
0,ode,0.117676,0.764594,4.759443,12.445251,2031,1,15,PC30,TrajectoryNet
0,ode,0.092073,0.778220,4.810399,12.521586,2031,1,15,PC30,TrajectoryNet


In [19]:
results = []
for repeat in ['model','model_run2','model_run3']:
    df = pd.read_csv(f"logs/TrajectoryNet/dm10/{repeat}/eval_combined.csv")
    results.append(df)

TJN_dm = pd.concat(results)
TJN_dm['space'] = 'DM10'
TJN_dm['method'] = 'TrajectoryNet'
TJN_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.129493,0.665401,3.010080,0.007514,2031,1,15,DM10,TrajectoryNet
0,ode,0.205810,0.687102,3.066409,0.007632,2031,1,15,DM10,TrajectoryNet
0,ode,0.129000,0.777138,2.866266,0.007135,2031,1,15,DM10,TrajectoryNet


# PRESICENT

In [20]:
results = [] 
for seed in [0,2,42]:
    df = pd.read_csv(f"results/PRESCIENT/pca_run/growth_weights-softplus_1_500-1e-06/seed_{seed}/eval_combined.csv")
    results.append(df)

PRESCIENT_pc = pd.concat(results)
PRESCIENT_pc['space'] = 'PC30'
PRESCIENT_pc['method'] = 'PRESCIENT'
PRESCIENT_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,sde,0.523880,0.897716,17.746177,17.746177,2031,100,15,PC30,PRESCIENT
0,sde,0.520433,0.899075,17.633522,17.633522,2031,100,15,PC30,PRESCIENT
0,sde,0.523387,0.900645,17.728345,17.728345,2031,100,15,PC30,PRESCIENT


In [21]:
results = [] 
for seed in [0,2,42]:
    df = pd.read_csv(f"results/PRESCIENT/dm_run/growth_weights-softplus_4_64-1e-06/seed_{seed}/eval_combined.csv")
    results.append(df)

PRESCIENT_dm = pd.concat(results)
PRESCIENT_dm['space'] = 'DM10'
PRESCIENT_dm['method'] = 'PRESCIENT'
PRESCIENT_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,sde,0.019202,-0.507419,0.497317,0.497317,2031,100,15,DM10,PRESCIENT
0,sde,0.019695,-0.511397,0.497644,0.497644,2031,100,15,DM10,PRESCIENT
0,sde,0.032496,-0.492015,0.496251,0.496251,2031,100,15,DM10,PRESCIENT


# SF2M 

In [22]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/sf2m/pca30/model_r{r}/eval_combined.csv"))

sf2m_pc = pd.concat(results)
sf2m_pc['space'] = 'PC30'
sf2m_pc['method'] = 'SF2M'
sf2m_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.263909,0.693820,4.014820,10.452632,2031,1,15,PC30,SF2M
1,sde,0.281142,0.703957,3.974596,10.296700,2031,100,15,PC30,SF2M
0,ode,0.263909,0.693820,4.014820,10.452632,2031,1,15,PC30,SF2M
1,sde,0.279173,0.703878,3.975438,10.296235,2031,100,15,PC30,SF2M
0,ode,0.263909,0.693820,4.014820,10.452632,2031,1,15,PC30,SF2M
1,sde,0.284097,0.704065,3.975195,10.299811,2031,100,15,PC30,SF2M


In [23]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/sf2m/dm10/model_r{r}/eval_combined.csv"))
sf2m_dm = pd.concat(results)
sf2m_dm['space'] = 'DM10'
sf2m_dm['method'] = 'SF2M'
sf2m_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.147218,0.024976,0.036775,0.000091,2031,1,15,DM10,SF2M
1,sde,0.306253,0.731162,0.030601,0.000073,2031,100,15,DM10,SF2M
0,ode,0.147218,0.024976,0.036775,0.000091,2031,1,15,DM10,SF2M
1,sde,0.315116,0.731319,0.030715,0.000074,2031,100,15,DM10,SF2M
0,ode,0.147218,0.024976,0.036775,0.000091,2031,1,15,DM10,SF2M
1,sde,0.314623,0.729017,0.030851,0.000074,2031,100,15,DM10,SF2M


# OTCFM

In [24]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/otcfm/pca30/model_r{r}/eval_combined.csv"))

otcfm_pc = pd.concat(results)
otcfm_pc['space'] = 'PC30'
otcfm_pc['method'] = 'OTCFM'
otcfm_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.312654,0.718214,4.138142,10.914734,2031,1,15,PC30,OTCFM
0,ode,0.312654,0.718214,4.138142,10.914734,2031,1,15,PC30,OTCFM
0,ode,0.312654,0.718214,4.138142,10.914734,2031,1,15,PC30,OTCFM


In [25]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/otcfm/dm10/model_r{r}/eval_combined.csv"))
otcfm_dm = pd.concat(results)
otcfm_dm['space'] = 'DM10'
otcfm_dm['method'] = 'OTCFM'
otcfm_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k,space,method
0,ode,0.372723,0.848451,2.521864,0.006332,2031,1,15,DM10,OTCFM
0,ode,0.372723,0.848451,2.521864,0.006332,2031,1,15,DM10,OTCFM
0,ode,0.372723,0.848451,2.521864,0.006332,2031,1,15,DM10,OTCFM


# Aggregate

In [ ]:
PC_fate_pdp = pd.read_csv("scripts/pdp_ranked_perform_PC.csv")
pdp_PC = pdp_PC.merge(PC_fate_pdp,  left_on=['model_dir'], right_on=['model_dir'])

DM_fate_pdp  = pd.read_csv("scripts/pdp_ranked_perform_DM.csv")
pdp_DM = pdp_DM.merge(DM_fate_pdp,  left_on=['model_dir'], right_on=['model_dir'])

In [ ]:
pdp_PC = pdp_PC.merge(PC_fate_pdp,  left_on=['model_dir'], right_on=['model_dir'])

In [ ]:
# pdp_PC
pc_df_list = [pdp_PC, DeepRUOT_pc, scDiffeq_pc, TIGON_pc,mioflow_pc,TJN_pc,PRESCIENT_pc,sf2m_pc,otcfm_pc]
common_col = set(DeepRUOT_pc.columns)
for i in range(len(pc_df_list)):
    common_col = common_col.intersection(set(pc_df_list[i].columns))
common_col = list(common_col)
common_col.remove('n_start_cells')
pc_performs = pd.concat([df.loc[:, common_col] for df in pc_df_list])
# pc_performs

In [101]:
dm_df_list = [pdp_DM, DeepRUOT_dm, scDiffeq_dm, TIGON_dm, mioflow_dm, TJN_dm, PRESCIENT_dm, sf2m_dm, otcfm_dm]
dm_performs = pd.concat([df.loc[:, common_col] for df in dm_df_list])

overall_performs = pd.concat([dm_performs, pc_performs], axis=0)

In [102]:
overall_performs

,accuracy,space,pearson_r,method,w2_raw
0,0.422944,DM10,0.861627,pseudodynamics+,0.007227
1,0.423929,DM10,0.750766,pseudodynamics+,0.007252
2,0.479074,DM10,0.911810,pseudodynamics+,0.007323
3,0.408173,DM10,0.844051,pseudodynamics+,0.007353
0,0.076809,DM10,0.809221,DeepRUOTv2,0.006889
...,...,...,...,...,...
0,0.263909,PC30,0.693820,SF2M,10.452632
1,0.284097,PC30,0.704065,SF2M,10.299811
0,0.312654,PC30,0.718214,OTCFM,10.914734
0,0.312654,PC30,0.718214,OTCFM,10.914734


In [104]:
overall_performs.to_csv("/rds/user/wz369/hpc-work/pseudodynamics_plus/results/benchmarking/klein_overall_performs.csv", index=False)